# 01 · GVMC wards + NAKSHA-Matcher (ML spatial matching)

**What this notebook does**
1. Downloads the open data for Visakhapatnam: OpenStreetMap streets and localities, Microsoft building footprints and Copernicus terrain.
2. Generates **all 98 GVMC ward zones**. They are clipped to the official Survey of India GVMC outline and named after real localities. No open dataset publishes the official ward boundaries, so the zones are marked *synthetic*.
3. Builds every ward's land-record layers: cadastre, property-tax GIS, revenue records, buildings and field GPS. The errors real departments make are injected, and an **answer key** records which parcel every record belongs to.
4. Trains **NAKSHA-Matcher**, a gradient-boosted classifier with isotonic calibration. For two records from different departments it predicts *are these the same land parcel?*, with a calibrated confidence.

| Model | Type | Solves (PS 26013) |
|---|---|---|
| NAKSHA-Matcher | Machine learning: calibrated classifier | AI/ML-based spatial matching · confidence scoring · feeds spatial conflict resolution |

**Runtime:** CPU is enough, no GPU needed. Turn **Internet on** (Kaggle: *Settings → Internet*). Building all 98 wards takes a while; set `WARDS` to a few ids for a quick run.

**Outputs** (zipped at the end):
- `match_model.joblib` + `match_model.meta.json`: the model and its metrics
- `wards.geojson`: the 98 ward zones
- `gvmc_buildings.geojson`: building labels for all wards, the input to notebook 02

Nothing is deployed.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
BRANCH = "main"          # git branch with the ML code
WARDS = None             # None = all 98 wards; e.g. "1,2,3" for a quick run
SYNTHETIC_BLOCKS = 200   # extra synthetic training blocks (0 = none)
SEED = 0

In [ ]:
# ── Environment: Kaggle, Colab or local ─────────────────────────────────────
import os, sys, subprocess, shutil, glob, json, time

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else os.path.abspath("naksha_work")
os.makedirs(WORK, exist_ok=True)
print("running on", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local", "| work dir:", WORK)


def run(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


# The code comes from the project repository (public). NAKSHA_REPO = use an existing checkout instead.
REPO_URL = "https://github.com/GVMC-NAKSHA-1/gvmc-naksha.git"
REPO = os.environ.get("NAKSHA_REPO") or os.path.join(WORK, "gvmc-naksha")
if not os.path.exists(os.path.join(REPO, ".git")) and not os.environ.get("NAKSHA_REPO"):
    run(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
    # Survey of India village boundaries (official GVMC outline) are stored with Git LFS.
    try:
        run("git lfs install --local && git lfs pull", cwd=REPO)
    except RuntimeError:
        print("Git LFS unavailable: wards will be clipped to the OpenStreetMap study area instead of the SoI outline.")
print("repo:", REPO)


In [ ]:
# ── Install (unpinned: works with the preinstalled Kaggle / Colab packages) ──
if not os.environ.get("NAKSHA_SKIP_INSTALL"):
    run(f"{sys.executable} -m pip install -q shapely pyproj rasterio pyshp gpxpy pillow psycopg2-binary boto3 redis "
        "scikit-learn joblib pandas matplotlib")
DATA_OUT, DATA_CACHE = os.path.join(WORK, "pack"), os.path.join(WORK, "cache")
OUTPUTS = os.path.join(WORK, "outputs")
os.makedirs(OUTPUTS, exist_ok=True)
PACK_ENV = {"DATA_OUT": DATA_OUT, "DATA_CACHE": DATA_CACHE, "PYTHONIOENCODING": "utf-8"}
DATA_DIR = os.path.join(REPO, "data")

## 1 · Download the open data (cached in `cache/`)

In [ ]:
run(f"{sys.executable} -m naksha_data fetch", cwd=DATA_DIR, env=PACK_ENV)

## 2 · Generate the 98 ward zones

In [ ]:
run(f"{sys.executable} -m naksha_data wards", cwd=DATA_DIR, env=PACK_ENV)
wards_fc = json.load(open(os.path.join(DATA_OUT, "wards.geojson")))
print(len(wards_fc["features"]), "wards")
shutil.copy(os.path.join(DATA_OUT, "wards.geojson"), OUTPUTS)

import matplotlib.pyplot as plt
from shapely.geometry import shape
fig, ax = plt.subplots(figsize=(8, 8))
for f in wards_fc["features"]:
    g = shape(f["geometry"])
    for poly in getattr(g, "geoms", [g]):
        ax.plot(*poly.exterior.xy, lw=0.6, color="#1d4f7c")
    c = g.representative_point()
    ax.annotate(str(f["properties"]["id"]), (c.x, c.y), fontsize=5, ha="center")
ax.set_title("GVMC ward zones (generated)"); ax.set_aspect("equal"); plt.show()

In [ ]:
import pandas as pd
pd.DataFrame([{"id": f["properties"]["id"], "name": f["properties"]["name"]} for f in wards_fc["features"]]).head(98)

## 3 · Build every ward's land-record layers (with the answer key)

In [ ]:
t0 = time.time()
run(f"{sys.executable} -m naksha_data build" + (f" --wards {WARDS}" if WARDS else ""), cwd=DATA_DIR, env=PACK_ENV)
print(f"built {len(glob.glob(os.path.join(DATA_OUT, 'ward-*')))} wards in {time.time() - t0:.0f}s")

## 4 · Train NAKSHA-Matcher on all built wards
The held-out test set is 20% of the **wards** (never random rows), so the scores measure performance on wards the model has not seen.

In [ ]:
MODEL = os.path.join(OUTPUTS, "match_model.joblib")
run(f"{sys.executable} src/train_matcher.py --pack {DATA_OUT} --blocks {SYNTHETIC_BLOCKS} --seed {SEED} "
    f"--no-db --force --out {MODEL}" + (f" --wards {WARDS}" if WARDS else ""),
    cwd=os.path.join(REPO, "worker"), env={"PYTHONIOENCODING": "utf-8"})

In [ ]:
meta = json.load(open(os.path.join(OUTPUTS, "match_model.meta.json")))
m = meta["metrics"]
rows = [{"scorer": "NAKSHA-Matcher (ML)", **m.get("model_on_same_rows", m["test"])}]
if "rule_baseline_test" in m:
    rows.append({"scorer": "current rules (geometry only)", **m["rule_baseline_test"]})
print("model", meta["version"], "| training data:", meta["sources"])
display(pd.DataFrame(rows).set_index("scorer"))
if m.get("test_by_ward"):
    by_ward = pd.DataFrame(m["test_by_ward"]).T
    print("held-out wards:", len(by_ward))
    display(by_ward[["pairs", "f1", "precision", "recall", "roc_auc"]].sort_values("f1"))

## 5 · Building labels for the deep-learning notebook (02)

In [ ]:
feats = []
for d in sorted(glob.glob(os.path.join(DATA_OUT, "ward-*"))):
    fp = os.path.join(d, "building_footprint.geojson")
    if os.path.exists(fp):
        feats += json.load(open(fp))["features"]
json.dump({"type": "FeatureCollection", "features": feats}, open(os.path.join(OUTPUTS, "gvmc_buildings.geojson"), "w"))
print(len(feats), "building footprints from", len(glob.glob(os.path.join(DATA_OUT, "ward-*"))), "wards")

In [ ]:
ARCHIVE_NAME = "naksha_matcher_outputs"
# ── Package the outputs ─────────────────────────────────────────────────────
archive = shutil.make_archive(os.path.join(WORK, ARCHIVE_NAME), "zip", OUTPUTS)
print("outputs:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
for f in sorted(os.listdir(OUTPUTS)):
    print("  ", f)
if ON_COLAB:
    from google.colab import files
    files.download(archive)
elif ON_KAGGLE:
    print("Kaggle: click 'Save Version' (Save & Run All) to keep /kaggle/working; download it from the Output tab.")
